<a href="https://colab.research.google.com/github/Rahat048/Batch-62/blob/main/LangChain_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -qU langchain-pinecone langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 4.1 MB/s eta 0:00:00


In [18]:
from google.colab import userdata

from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = userdata.get('PINECONE_API_KEY')

pc = Pinecone(api_key=pinecone_api_key)

In [19]:
index_name = "langchain-rag-project"

pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [76]:
import getpass
import os
from google.colab import userdata

GEMINI_KEY = userdata.get('GOOGLE_API_KEY')

os.environ["GOOGLE_API_KEY"] = GEMINI_KEY

In [77]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [78]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [89]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="He is head of department of tolling development and gauge control. He is also Deputy manager.",
    metadata={"name": "Abbas"},
)

document_2 = Document(
    page_content="He is calibration inspecor in gauge control.",
    metadata={"name": "Hassan"},
)

document_3 = Document(
    page_content="He is class 3 cmm operator in gauge control. ",
    metadata={"name": "Raees"},
)

document_4 = Document(
    page_content="He is class 1 cmm operator in gauge control.",
    metadata={"name": "Shah"},
)

document_5 = Document(
    page_content="He is calibration inspector in gauge control. He is now a team leader in gauge control. He is most senior in gauge control.",
    metadata={"name": "A.Baksh"},
)

document_6 = Document(
    page_content="He is calibration inspector in gauge control. He is now a team leader in gauge control. He is junior than A.Baksh in gauge control.",
    metadata={"name": "Sajid"},
)

document_7 = Document(
    page_content="He is cmm operator in gauge control. He is also a team leader in gauge control.",
    metadata={"name": "Rahat"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['f4acbc67-9192-496d-b8fa-044b922f5131',
 'ca173b33-3323-416d-bae1-02d14a0c1430',
 'ebee07a2-b25c-4291-9a75-ba82a7abd0f0',
 '7c14430a-611a-4a49-9bf7-a3d0c952ae1b',
 '8d72ad44-9a93-464e-97c3-bdcb47be1f3b',
 'e52d05bb-a204-4fe9-a2ab-b22b2c4d9eca',
 '4fc7d85d-36f6-4ae7-8424-96c323f83aba']

In [90]:
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.prompts import PromptTemplate

In [91]:
def user_answer(question: str):

  vector_results = vector_store.similarity_search(question, k=7)

  llm = ChatGoogleGenerativeAI(
      api_key=GEMINI_KEY,
      model="gemini-2.0-flash-exp",
      temperature=0.7,
      max_tokens= 100
  )

  prompt1 = PromptTemplate(
      input_variables=["question"],
      template="Using this data {vector_results}. Answer the following question:\n\n{question}."
  )

  chain1 = prompt1 | llm

  final_answer = chain1.invoke({"question": question, "vector_results": vector_results})

  return final_answer

In [96]:
response = user_answer("who is abbas?")

In [97]:
print("Answer:", response.content)

Answer: Based on the provided data, Abbas is the head of the department of tolling development and gauge control. He is also a Deputy manager.

